# Cube Kernel Walkthrough

The cube kernel lives entirely in `cube_kernel.transforms`. No coordinates, no solvers — just NumPy integer matrices.

Two layers:
- **Single cubie**: a 3×3 orientation matrix. Identity = solved.
- **Three cubies**: a 9×3 matrix — three 3×3 orientation blocks stacked. Tiled identity = all solved.

One rule applies throughout: `new_state = T @ state`.

In [47]:
import numpy as np

from cube_kernel.transforms import (
    TRANSFORMS,
    cycle_bwd,
    cycle_fwd,
    orient_at_0,
    orient_at_1,
    orient_at_2,
    swap_0_1,
    swap_0_2,
    swap_1_2,
    x_minus_90,
    x_plus_90,
    y_minus_90,
    y_plus_90,
    z_minus_90,
    z_plus_90,
)
from cube_kernel.pretty import show, show_block, show_cubie_state, show_equation

---
## Part 1 — Single Cubie

A cubie's orientation is a 3×3 integer matrix. Each column is a world-space unit vector showing where one of the cubie's local axes currently points. Identity means perfectly aligned with the world frame.

### The solved state

In [48]:
I = np.eye(3, dtype=int)
show(I, label="identity — every local axis aligns with its world axis")

identity — every local axis aligns with its world axis
┌──────────┐
│  1  0  0 │
│  0  1  0 │
│  0  0  1 │
└──────────┘


### The six quarter-turn rotations

Each rotation is a 3×3 matrix derived from the right-hand rule. Negative rotations are the transposes of their positive counterparts — a free consequence of rotation matrices being orthogonal (R⁻¹ = Rᵀ).

In [49]:
for name, t in TRANSFORMS.items():
    show(t, label=name)
    print()

x+
┌──────────┐
│  1  0  0 │
│  0  0 -1 │
│  0  1  0 │
└──────────┘

x-
┌──────────┐
│  1  0  0 │
│  0  0  1 │
│  0 -1  0 │
└──────────┘

y+
┌──────────┐
│  0  0  1 │
│  0  1  0 │
│ -1  0  0 │
└──────────┘

y-
┌──────────┐
│  0  0 -1 │
│  0  1  0 │
│  1  0  0 │
└──────────┘

z+
┌──────────┐
│  0 -1  0 │
│  1  0  0 │
│  0  0  1 │
└──────────┘

z-
┌──────────┐
│  0  1  0 │
│ -1  0  0 │
│  0  0  1 │
└──────────┘



### Negative rotations are transposes

In [50]:
show_equation(x_minus_90(), x_plus_90().T, ops=["="], label="x-  =  (x+).T")
print()
show_equation(y_minus_90(), y_plus_90().T, ops=["="], label="y-  =  (y+).T")

x-  =  (x+).T
┌──────────┐     ┌──────────┐
│  1  0  0 │     │  1  0  0 │
│  0  0  1 │  =  │  0  0  1 │
│  0 -1  0 │     │  0 -1  0 │
└──────────┘     └──────────┘

y-  =  (y+).T
┌──────────┐     ┌──────────┐
│  0  0 -1 │     │  0  0 -1 │
│  0  1  0 │  =  │  0  1  0 │
│  1  0  0 │     │  1  0  0 │
└──────────┘     └──────────┘


### Applying a transform — left-multiplication

Applying rotation T to state S means `T @ S`.

In [51]:
T = x_plus_90()
S = np.eye(3, dtype=int)
show_equation(T, S, T @ S, ops=["@", "="], label="x+  @  I  =  x+")

x+  @  I  =  x+
┌──────────┐     ┌──────────┐     ┌──────────┐
│  1  0  0 │     │  1  0  0 │     │  1  0  0 │
│  0  0 -1 │  @  │  0  1  0 │  =  │  0  0 -1 │
│  0  1  0 │     │  0  0  1 │     │  0  1  0 │
└──────────┘     └──────────┘     └──────────┘


### Composing transforms

Chaining transforms is just repeated left-multiplication. Four quarter-turns around the same axis return to identity.

In [52]:
state = np.eye(3, dtype=int)
for step in range(5):
    show(state, label=f"x+ applied {step}× {'← identity' if step == 4 else ''}")
    print()
    state = x_plus_90() @ state

x+ applied 0× 
┌──────────┐
│  1  0  0 │
│  0  1  0 │
│  0  0  1 │
└──────────┘

x+ applied 1× 
┌──────────┐
│  1  0  0 │
│  0  0 -1 │
│  0  1  0 │
└──────────┘

x+ applied 2× 
┌──────────┐
│  1  0  0 │
│  0 -1  0 │
│  0  0 -1 │
└──────────┘

x+ applied 3× 
┌──────────┐
│  1  0  0 │
│  0  0  1 │
│  0 -1  0 │
└──────────┘

x+ applied 4× ← identity
┌──────────┐
│  1  0  0 │
│  0  1  0 │
│  0  0  1 │
└──────────┘



Two non-commuting transforms — order matters:

In [53]:
xy = y_plus_90() @ x_plus_90() @ I
yx = x_plus_90() @ y_plus_90() @ I
show_equation(xy, yx, ops=["≠"], label="(y+ ∘ x+) ≠ (x+ ∘ y+)")

(y+ ∘ x+) ≠ (x+ ∘ y+)
┌──────────┐     ┌──────────┐
│  0  1  0 │     │  0  0  1 │
│  0  0 -1 │  ≠  │  1  0  0 │
│ -1  0  0 │     │  0  1  0 │
└──────────┘     └──────────┘


---
## Part 2 — Three-Cubie State

The state for three cubies is a **9×3 matrix**: three 3×3 orientation blocks stacked vertically. Cubie `k` occupies rows `3k … 3k+2`.

The same `T @ state` rule still applies. The only change is that `T` is now a **9×9 block matrix**.

### Initial state — all cubies solved

In [54]:
initial = np.tile(np.eye(3, dtype=int), (3, 1))
show_cubie_state(initial, label="initial state")

initial state
  Cubie 0          Cubie 1          Cubie 2   
┌──────────┐     ┌──────────┐     ┌──────────┐
│  1  0  0 │     │  1  0  0 │     │  1  0  0 │
│  0  1  0 │     │  0  1  0 │     │  0  1  0 │
│  0  0  1 │     │  0  0  1 │     │  0  0  1 │
└──────────┘     └──────────┘     └──────────┘


### Orienting one cubie — block-diagonal (9×9) matrices

`orient_at_k(t)` builds a 9×9 block-diagonal matrix: rotation `t` sits in block `k`, identity fills the other two blocks. Applying it with `@` rotates only cubie `k`.

In [55]:
show_block(orient_at_0(x_plus_90()), label="orient_at_0(x+)")

orient_at_0(x+)
┌──────────┬──────────┬──────────┐
│  1  0  0 │  0  0  0 │  0  0  0 │
│  0  0 -1 │  0  0  0 │  0  0  0 │
│  0  1  0 │  0  0  0 │  0  0  0 │
├──────────┼──────────┼──────────┤
│  0  0  0 │  1  0  0 │  0  0  0 │
│  0  0  0 │  0  1  0 │  0  0  0 │
│  0  0  0 │  0  0  1 │  0  0  0 │
├──────────┼──────────┼──────────┤
│  0  0  0 │  0  0  0 │  1  0  0 │
│  0  0  0 │  0  0  0 │  0  1  0 │
│  0  0  0 │  0  0  0 │  0  0  1 │
└──────────┴──────────┴──────────┘


In [56]:
state = orient_at_0(x_plus_90()) @ initial
show_cubie_state(state, label="after orient_at_0(x+)")

after orient_at_0(x+)
  Cubie 0          Cubie 1          Cubie 2   
┌──────────┐     ┌──────────┐     ┌──────────┐
│  1  0  0 │     │  1  0  0 │     │  1  0  0 │
│  0  0 -1 │     │  0  1  0 │     │  0  1  0 │
│  0  1  0 │     │  0  0  1 │     │  0  0  1 │
└──────────┘     └──────────┘     └──────────┘


In [57]:
state = orient_at_1(z_minus_90()) @ state
show_cubie_state(state, label="then orient_at_1(z-)")

then orient_at_1(z-)
  Cubie 0          Cubie 1          Cubie 2   
┌──────────┐     ┌──────────┐     ┌──────────┐
│  1  0  0 │     │  0  1  0 │     │  1  0  0 │
│  0  0 -1 │     │ -1  0  0 │     │  0  1  0 │
│  0  1  0 │     │  0  0  1 │     │  0  0  1 │
└──────────┘     └──────────┘     └──────────┘


### Permuting cubie positions — block permutation (9×9) matrices

Swaps and cycles are 9×9 block permutation matrices. Each one moves entire 3×3 cubie blocks to new slots. Same `T @ state` rule.

In [58]:
show_block(swap_0_1(), label="swap_0_1  — swaps cubies 0 and 1")

swap_0_1  — swaps cubies 0 and 1
┌──────────┬──────────┬──────────┐
│  0  0  0 │  1  0  0 │  0  0  0 │
│  0  0  0 │  0  1  0 │  0  0  0 │
│  0  0  0 │  0  0  1 │  0  0  0 │
├──────────┼──────────┼──────────┤
│  1  0  0 │  0  0  0 │  0  0  0 │
│  0  1  0 │  0  0  0 │  0  0  0 │
│  0  0  1 │  0  0  0 │  0  0  0 │
├──────────┼──────────┼──────────┤
│  0  0  0 │  0  0  0 │  1  0  0 │
│  0  0  0 │  0  0  0 │  0  1  0 │
│  0  0  0 │  0  0  0 │  0  0  1 │
└──────────┴──────────┴──────────┘


In [59]:
show_cubie_state(state, label="before swap")
print()
show_cubie_state(swap_0_1() @ state, label="after swap_0_1")

before swap
  Cubie 0          Cubie 1          Cubie 2   
┌──────────┐     ┌──────────┐     ┌──────────┐
│  1  0  0 │     │  0  1  0 │     │  1  0  0 │
│  0  0 -1 │     │ -1  0  0 │     │  0  1  0 │
│  0  1  0 │     │  0  0  1 │     │  0  0  1 │
└──────────┘     └──────────┘     └──────────┘

after swap_0_1
  Cubie 0          Cubie 1          Cubie 2   
┌──────────┐     ┌──────────┐     ┌──────────┐
│  0  1  0 │     │  1  0  0 │     │  1  0  0 │
│ -1  0  0 │     │  0  0 -1 │     │  0  1  0 │
│  0  0  1 │     │  0  1  0 │     │  0  0  1 │
└──────────┘     └──────────┘     └──────────┘


In [60]:
show_block(cycle_fwd(), label="cycle_fwd  — 0 → 1 → 2 → 0")

cycle_fwd  — 0 → 1 → 2 → 0
┌──────────┬──────────┬──────────┐
│  0  0  0 │  0  0  0 │  1  0  0 │
│  0  0  0 │  0  0  0 │  0  1  0 │
│  0  0  0 │  0  0  0 │  0  0  1 │
├──────────┼──────────┼──────────┤
│  1  0  0 │  0  0  0 │  0  0  0 │
│  0  1  0 │  0  0  0 │  0  0  0 │
│  0  0  1 │  0  0  0 │  0  0  0 │
├──────────┼──────────┼──────────┤
│  0  0  0 │  1  0  0 │  0  0  0 │
│  0  0  0 │  0  1  0 │  0  0  0 │
│  0  0  0 │  0  0  1 │  0  0  0 │
└──────────┴──────────┴──────────┘


In [61]:
show_cubie_state(state, label="before cycle")
print()
show_cubie_state(cycle_fwd() @ state, label="after cycle_fwd")
print()
show_cubie_state(cycle_bwd() @ cycle_fwd() @ state, label="cycle_bwd undoes cycle_fwd")

before cycle
  Cubie 0          Cubie 1          Cubie 2   
┌──────────┐     ┌──────────┐     ┌──────────┐
│  1  0  0 │     │  0  1  0 │     │  1  0  0 │
│  0  0 -1 │     │ -1  0  0 │     │  0  1  0 │
│  0  1  0 │     │  0  0  1 │     │  0  0  1 │
└──────────┘     └──────────┘     └──────────┘

after cycle_fwd
  Cubie 0          Cubie 1          Cubie 2   
┌──────────┐     ┌──────────┐     ┌──────────┐
│  1  0  0 │     │  1  0  0 │     │  0  1  0 │
│  0  1  0 │     │  0  0 -1 │     │ -1  0  0 │
│  0  0  1 │     │  0  1  0 │     │  0  0  1 │
└──────────┘     └──────────┘     └──────────┘

cycle_bwd undoes cycle_fwd
  Cubie 0          Cubie 1          Cubie 2   
┌──────────┐     ┌──────────┐     ┌──────────┐
│  1  0  0 │     │  0  1  0 │     │  1  0  0 │
│  0  0 -1 │     │ -1  0  0 │     │  0  1  0 │
│  0  1  0 │     │  0  0  1 │     │  0  0  1 │
└──────────┘     └──────────┘     └──────────┘


### Composing operations

Every operation — orientation and permutation — is a matrix. Chain them freely with `@`. The kernel does not distinguish between the two kinds of transform.

In [62]:
# Orient cubie 0 with x+, then cubie 2 with y+, then cycle everything forward.
composed = cycle_fwd() @ orient_at_2(y_plus_90()) @ orient_at_0(x_plus_90())

show_cubie_state(initial, label="initial")
print()
show_cubie_state(composed @ initial, label="orient_at_0(x+)  →  orient_at_2(y+)  →  cycle_fwd")

initial
  Cubie 0          Cubie 1          Cubie 2   
┌──────────┐     ┌──────────┐     ┌──────────┐
│  1  0  0 │     │  1  0  0 │     │  1  0  0 │
│  0  1  0 │     │  0  1  0 │     │  0  1  0 │
│  0  0  1 │     │  0  0  1 │     │  0  0  1 │
└──────────┘     └──────────┘     └──────────┘

orient_at_0(x+)  →  orient_at_2(y+)  →  cycle_fwd
  Cubie 0          Cubie 1          Cubie 2   
┌──────────┐     ┌──────────┐     ┌──────────┐
│  0  0  1 │     │  1  0  0 │     │  1  0  0 │
│  0  1  0 │     │  0  0 -1 │     │  0  1  0 │
│ -1  0  0 │     │  0  1  0 │     │  0  0  1 │
└──────────┘     └──────────┘     └──────────┘


In [63]:
# A composed transform is itself a matrix — inspect its block structure.
show_block(composed, label="composed transform (9×9)")

composed transform (9×9)
┌──────────┬──────────┬──────────┐
│  0  0  0 │  0  0  0 │  0  0  1 │
│  0  0  0 │  0  0  0 │  0  1  0 │
│  0  0  0 │  0  0  0 │ -1  0  0 │
├──────────┼──────────┼──────────┤
│  1  0  0 │  0  0  0 │  0  0  0 │
│  0  0 -1 │  0  0  0 │  0  0  0 │
│  0  1  0 │  0  0  0 │  0  0  0 │
├──────────┼──────────┼──────────┤
│  0  0  0 │  1  0  0 │  0  0  0 │
│  0  0  0 │  0  1  0 │  0  0  0 │
│  0  0  0 │  0  0  1 │  0  0  0 │
└──────────┴──────────┴──────────┘
